In [ ]:
# An example of adapting the code for HW1P2 to our project
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision
# Enable mixed precision for faster kernels on A100 (may improve throughput)
mixed_precision.set_global_policy('mixed_float16')
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.models import load_model

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# load iris
iris = load_iris()
X = iris.data.astype("float32")
y = iris.target.astype("int32")
num_classes = len(np.unique(y))

# one-hot label
y_onehot = tf.keras.utils.to_categorical(y, num_classes)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features to [0,1] for meaningful CF optimization
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [16]:
# train a decision tree
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [17]:
y_pred = tree.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('accuracy:', acc)

accuracy: 1.0


In [18]:

sk_tree = tree.tree_

print("n_nodes:", sk_tree.node_count)
print("children_left:", sk_tree.children_left)
print("children_right:", sk_tree.children_right)
print("feature:", sk_tree.feature)
print("threshold:", sk_tree.threshold)
print("value (class counts):")
print(sk_tree.value)

n_nodes: 9
children_left: [ 1 -1  3  4 -1 -1  7 -1 -1]
children_right: [ 2 -1  6  5 -1 -1  8 -1 -1]
feature: [ 2 -2  2  3 -2 -2  3 -2 -2]
threshold: [ 2.44999999 -2.          4.75        1.60000002 -2.         -2.
  1.75       -2.         -2.        ]
value (class counts):
[[[0.2952381  0.35238095 0.35238095]]

 [[1.         0.         0.        ]]

 [[0.         0.5        0.5       ]]

 [[0.         0.96969697 0.03030303]]

 [[0.         1.         0.        ]]

 [[0.         0.         1.        ]]

 [[0.         0.12195122 0.87804878]]

 [[0.         0.5        0.5       ]]

 [[0.         0.03030303 0.96969697]]]


In [19]:
from sklearn.tree import export_text
print(export_text(tree, feature_names=iris.feature_names))


|--- petal length (cm) <= 2.45
|   |--- class: 0
|--- petal length (cm) >  2.45
|   |--- petal length (cm) <= 4.75
|   |   |--- petal width (cm) <= 1.60
|   |   |   |--- class: 1
|   |   |--- petal width (cm) >  1.60
|   |   |   |--- class: 2
|   |--- petal length (cm) >  4.75
|   |   |--- petal width (cm) <= 1.75
|   |   |   |--- class: 1
|   |   |--- petal width (cm) >  1.75
|   |   |   |--- class: 2



In [20]:
def print_custom_tree(node, depth=0):
    indent = "    " * depth

    # Leaf node
    if node.value is not None:
        print(f"{indent}Leaf: value={node.value.numpy()}")
        return

    # Internal node
    print(f"{indent}Node: feature={node.feature}, threshold={node.threshold}")
    
    # Print children
    print(f"{indent} L->")
    print_custom_tree(node.left, depth + 1)

    print(f"{indent} R->")
    print_custom_tree(node.right, depth + 1)



In [21]:
# 3. Build soft tree

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # probability vector (leaf)

def build_tree(sklearn_tree, node_id=0):
    # leaf node
    if sklearn_tree.children_left[node_id] == sklearn_tree.children_right[node_id]:
        # sklearn gives shape (1,3)
        value = sklearn_tree.value[node_id][0]
        value = value / value.sum()
        value = tf.constant(value, dtype=tf.float32)   # fixed tensor
        return Node(value=value)

    # inner node
    feature = sklearn_tree.feature[node_id]
    threshold = sklearn_tree.threshold[node_id]
    left_child = build_tree(sklearn_tree, sklearn_tree.children_left[node_id])
    right_child = build_tree(sklearn_tree, sklearn_tree.children_right[node_id])
    return Node(feature=feature, threshold=threshold, left=left_child, right=right_child)

soft_tree_root = build_tree(tree.tree_)
print_custom_tree(soft_tree_root)

Node: feature=2, threshold=2.449999988079071
 L->
    Leaf: value=[1. 0. 0.]
 R->
    Node: feature=2, threshold=4.75
     L->
        Node: feature=3, threshold=1.600000023841858
         L->
            Leaf: value=[0. 1. 0.]
         R->
            Leaf: value=[0. 0. 1.]
     R->
        Node: feature=3, threshold=1.75
         L->
            Leaf: value=[0.  0.5 0.5]
         R->
            Leaf: value=[0.         0.03030303 0.969697  ]


In [ ]:
# 4. Soft prediction
DEFAULT_ALPHA = 20   # steeper sigmoid improves approximation

def soft_tree_predict(node, x, alpha):
    if node.value is not None:
        # IMPORTANT: return a *copy* to avoid broadcasting bugs
        return tf.identity(node.value)

    # s = tf.sigmoid(alpha * (x[node.feature] - node.threshold))
    s = tf.sigmoid(alpha * (node.threshold - x[node.feature]))
    left_val = soft_tree_predict(node.left, x, alpha)
    right_val = soft_tree_predict(node.right, x, alpha)

    return s * left_val + (1 - s) * right_val

def predict_batch(root, X, alpha=DEFAULT_ALPHA):
    # accept numpy arrays or tensors and ensure float32 dtype for numerical stability
    X_tf = tf.convert_to_tensor(X)
    if X_tf.dtype != tf.float32:
        X_tf = tf.cast(X_tf, tf.float32)
    return tf.vectorized_map(lambda x: soft_tree_predict(root, x, alpha), X_tf)

# ---------------------------
# 5. Test accuracy
# ---------------------------
y_pred_soft = predict_batch(soft_tree_root, X_test).numpy()
y_pred_cls = np.argmax(y_pred_soft, axis=1)

acc = (y_pred_cls == y_test).mean()
print("Soft-tree approx accuracy:", acc)
print(y_pred_cls)
print(y_test)
y_pred_dr = tree.predict(X_test)
print(y_pred_dr)

Soft-tree approx accuracy: 0.9555555555555556
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 2 2 1 1 2 0 2 0 2 2 2 2 2 0 0 0 0 1 0 0 2 1
 0 0 0 2 2 1 0 0]
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0 0 0 1 0 0 2 1
 0 0 0 2 1 1 0 0]
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0 0 0 1 0 0 2 1
 0 0 0 2 1 1 0 0]


In [ ]:
# Batched counterfactual search using a cached solver object per batch-shape
# The solver class creates tf.Variables (Adam state and x_cf) once (in Python scope)
# and exposes a compiled `solve` method. This avoids creating Variables inside
# a tf.function on repeated calls while keeping Adam optimizer for speed.
_cf_solver_cache = {}

class _BatchedCFSolver:
    def __init__(self, surrogate, B, D, steps, lr, l2, alpha_cf, tol):
        self.surrogate = surrogate
        self.B = B
        self.D = D
        self.steps = int(steps)
        self.lr = float(lr)
        self.l2 = float(l2)
        self.alpha_cf = float(alpha_cf)
        self.tol = float(tol)
        # create variables once
        self.x_cf = tf.Variable(tf.zeros([B, D], dtype=tf.float32), trainable=True)
        self.opt = tf.keras.optimizers.Adam(learning_rate=self.lr)

        @tf.function(input_signature=[tf.TensorSpec([B, D], tf.float32), tf.TensorSpec([B], tf.int32)])
        def _solve(x0, target_cls):
            # initialize x_cf from x0
            self.x_cf.assign(x0)
            target_onehot = tf.one_hot(target_cls, self.surrogate.num_classes, dtype=tf.float32)
            for _ in tf.range(self.steps):
                with tf.GradientTape() as tape:
                    probs = self.surrogate.call(self.x_cf, alpha=self.alpha_cf)
                    p_t = tf.reduce_sum(probs * target_onehot, axis=1)
                    loss_per = -tf.math.log(tf.clip_by_value(p_t, 1e-8, 1.0)) + self.l2 * tf.reduce_sum(tf.square(self.x_cf - x0), axis=1)
                    loss = tf.reduce_mean(loss_per)
                grads = tape.gradient(loss, self.x_cf)
                self.opt.apply_gradients([(grads, self.x_cf)])
                self.x_cf.assign(tf.clip_by_value(self.x_cf, 0.0, 1.0))
                if tf.reduce_all(p_t >= self.tol):
                    break
            return self.x_cf

        self._solve = _solve

    def solve(self, x0, target_cls):
        x0_t = tf.convert_to_tensor(np.asarray(x0, dtype=np.float32))
        target_t = tf.convert_to_tensor(np.asarray(target_cls, dtype=np.int32), dtype=tf.int32)
        return self._solve(x0_t, target_t)

def find_counterfactual_surrogate_batched(surrogate, X_seeds, target_cls, steps=100, lr=0.04, l2=1.0, alpha_cf=DEFAULT_ALPHA, tol=0.99):
    X_np = np.asarray(X_seeds, dtype=np.float32)
    B, D = X_np.shape
    key = (B, D, int(steps), float(lr), float(l2), float(alpha_cf), float(tol))
    if key not in _cf_solver_cache:
        _cf_solver_cache[key] = _BatchedCFSolver(surrogate, B, D, steps, lr, l2, alpha_cf, tol)
    solver = _cf_solver_cache[key]
    return solver.solve(X_np, target_cls)

In [ ]:
def compute_fidelity(model_original,model_surrogate,X_test):
  y_predict_original = model_original(X_test)
  y_predict_original_idx = tf.math.argmax(y_predict_original, axis=1)
  y_predict_surrogate = model_surrogate(X_test)
  y_predict_surrogate_idx = tf.math.argmax(y_predict_surrogate, axis=1)
  comparison = tf.equal(y_predict_original_idx, y_predict_surrogate_idx)
  num_same = int(tf.reduce_sum(tf.cast(comparison, tf.int32)))
  fidelity = num_same/X_test.shape[0]
  return fidelity

class SklearnTreeWrapper(tf.keras.Model):
    def __init__(self, clf):
        super().__init__()
        self.clf = clf

    def call(self, inputs):
        # 转成 numpy，调用 sklearn 的 predict_proba
        inputs_np = inputs.numpy().astype(np.float32)
        y_proba = self.clf.predict_proba(inputs_np)

        # 重新变成 TensorFlow 张量
        return tf.convert_to_tensor(y_proba, dtype=tf.float32)


class SoftTreeModel(tf.keras.Model):
    def __init__(self, root, alpha=DEFAULT_ALPHA):
        super().__init__()
        self.root = root
        self.alpha = alpha

    def call(self, inputs):
        return predict_batch(self.root, inputs, alpha=self.alpha)

In [ ]:
model_original = SklearnTreeWrapper(tree)
soft_tree_model_static = SoftTreeModel(soft_tree_root, alpha=DEFAULT_ALPHA)
# compute fidelity between sklearn wrapper and the static soft-tree approximation
f = compute_fidelity(model_original, soft_tree_model_static, X_test)
print('fidelity (static soft approximate):', f)


fidelity: 0.9555555555555556


In [ ]:
# --- Parametric soft-tree surrogate, counterfactual clamping and FOCUS querying ---
import copy
from collections import deque
import numpy as np
import tensorflow as tf

class ParametricSoftTree(tf.keras.Model):
    def __init__(self, sklearn_tree, num_features, num_classes, init_from_sklearn=True):
        super().__init__()
        self.num_features = num_features
        self.num_classes = num_classes
        # extract tree structure
        sk = sklearn_tree
        self.children_left = np.array(sk.children_left)
        self.children_right = np.array(sk.children_right)
        self.feature = np.array(sk.feature)
        self.threshold = np.array(sk.threshold)
        self.node_count = int(sk.node_count)
        # identify leaves and map node->leaf_idx
        is_leaf = (self.children_left == self.children_right)
        self.is_leaf = is_leaf
        self.node_to_leaf = {}
        leaf_idx = 0
        for nid in range(self.node_count):
            if is_leaf[nid]:
                self.node_to_leaf[nid] = leaf_idx
                leaf_idx += 1
        self.n_leaves = leaf_idx
        # trainable thresholds for every node (only used for internal nodes)
        init_thresh = np.where(np.isfinite(self.threshold), self.threshold, 0.0).astype(np.float32)
        self.threshold_var = self.add_weight(shape=(self.node_count,), initializer=tf.constant_initializer(init_thresh), trainable=True, dtype=tf.float32)
        # leaf logits (trainable)
        # initialize from sklearn leaf value distributions if requested
        if init_from_sklearn:
            leaf_logits = np.zeros((self.n_leaves, num_classes), dtype=np.float32)
            for nid in range(self.node_count):
                if is_leaf[nid]:
                    li = self.node_to_leaf[nid]
                    vals = sk.value[nid][0].astype(np.float32)
                    probs = vals / (vals.sum() + 1e-12)
                    # convert to logits safe-guarded
                    leaf_logits[li] = np.log(probs + 1e-8)
        else:
            leaf_logits = np.random.randn(self.n_leaves, num_classes).astype(np.float32) * 0.1
        self.leaf_logits = self.add_weight(shape=(self.n_leaves, num_classes), initializer=tf.constant_initializer(leaf_logits), trainable=True, dtype=tf.float32)

    def call(self, inputs, alpha=DEFAULT_ALPHA):
        # inputs: [batch, num_features]
        inputs = tf.convert_to_tensor(inputs)
        if inputs.dtype != tf.float32:
            inputs = tf.cast(inputs, tf.float32)
        def predict_one(x):
            return self._predict_node(0, x, alpha)
        return tf.vectorized_map(predict_one, inputs)

    def _predict_node(self, node_id, x, alpha):
        if self.is_leaf[node_id]:
            leaf_idx = self.node_to_leaf[node_id]
            logits = self.leaf_logits[leaf_idx]
            return tf.nn.softmax(logits)
        feat = int(self.feature[node_id])
        thresh = self.threshold_var[node_id]
        # sigmoid decision (note sign to match earlier implementation)
        s = tf.sigmoid(alpha * (thresh - x[feat]))
        left_val = self._predict_node(int(self.children_left[node_id]), x, alpha)
        right_val = self._predict_node(int(self.children_right[node_id]), x, alpha)
        return s * left_val + (1.0 - s) * right_val

    def predict_proba(self, X, alpha=DEFAULT_ALPHA):
        return self.call(X, alpha=alpha).numpy()


/Users/hoffmann/Documents/Development/tree-based-model-approx/.venv/lib/python3.13/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


KeyboardInterrupt: 

In [ ]:
# --- Counterfactual search, FOCUS queries, surrogate training, and reconstruction ---
import time
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import tensorflow as tf

def find_counterfactual_surrogate(surrogate, x_orig, target_class_idx, steps=200, lr=0.02, l2=1.0, alpha_cf=DEFAULT_ALPHA, tol=0.99):
    # x_orig: 1D numpy array in feature space (assumed scaled to [0,1])
    x0 = x_orig.astype(np.float32).reshape(1, -1)
    x_cf = tf.Variable(x0.copy(), dtype=tf.float32)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    for step in range(steps):
        with tf.GradientTape() as tape:
            probs = surrogate.call(x_cf, alpha=alpha_cf)[0]
            p_t = tf.clip_by_value(probs[target_class_idx], 1e-8, 1.0)
            loss = -tf.math.log(p_t) + l2 * tf.reduce_sum(tf.square(x_cf - x0))
        grads = tape.gradient(loss, x_cf)
        opt.apply_gradients([(grads, x_cf)])
        # clamp to [0,1] assuming scaled inputs
        x_cf.assign(tf.clip_by_value(x_cf, 0.0, 1.0))
        if float(probs[target_class_idx]) >= tol:
            break
    return x_cf.numpy().reshape(-1)

def generate_focus_queries(surrogate, X_pool, n_queries, batch_size=32, alpha_cf=DEFAULT_ALPHA, rng_seed=0):
    rng = np.random.RandomState(rng_seed)
    X_pool = np.asarray(X_pool)
    queries = []
    pool_idx = np.arange(len(X_pool))
    while len(queries) < n_queries:
        # sample seeds without replacement where possible
        choose = rng.choice(pool_idx, size=min(batch_size, len(pool_idx)), replace=False)
        for i in choose:
            x = X_pool[i:i+1]
            proba = surrogate.predict_proba(x, alpha=alpha_cf)[0]
            pred_cls = int(np.argmax(proba))
            # choose different target class (wrap-around)
            target_cls = (pred_cls + 1) % surrogate.num_classes
            xcf = find_counterfactual_surrogate(surrogate, x.flatten(), target_cls, steps=200, lr=0.02, l2=1.0, alpha_cf=alpha_cf)
            queries.append(xcf)
            if len(queries) >= n_queries:
                break
        # if pool smaller than needed, allow reselection next iteration
        if len(pool_idx) > batch_size:
            pool_idx = np.setdiff1d(pool_idx, choose)
        else:
            pool_idx = np.arange(len(X_pool))
    return np.stack(queries, axis=0)

def train_surrogate_on_queries(surrogate, X_q, y_q_proba, epochs=10, batch_size=32, alpha=DEFAULT_ALPHA, lr=1e-3):
    X_q = np.asarray(X_q, dtype=np.float32)
    y_q_proba = np.asarray(y_q_proba, dtype=np.float32)
    dataset = tf.data.Dataset.from_tensor_slices((X_q, y_q_proba)).shuffle(1000).batch(batch_size)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=False)
    for epoch in range(epochs):
        epoch_loss = 0.0
        n = 0
        for xb, yb in dataset:
            with tf.GradientTape() as tape:
                preds = surrogate.call(xb, alpha=alpha)
                loss = loss_fn(yb, preds)
            grads = tape.gradient(loss, surrogate.trainable_variables)
            opt.apply_gradients(zip(grads, surrogate.trainable_variables))
            epoch_loss += float(loss) * xb.shape[0]
            n += xb.shape[0]
        if n > 0:
            print(f'epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f}')

def reconstruct_tree_from_queries(X_q, y_q_proba, max_depth=3, random_state=42):
    # Train a sklearn DecisionTreeClassifier on hard labels from queried oracle probabilities
    y_hard = np.argmax(y_q_proba, axis=1)
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state)
    clf.fit(X_q, y_hard)
    return clf

def run_reconstruction_experiment(tree_oracle, X_train, X_test, query_budgets=[50,100,200,500], batch_size=32, seed_size=20):
    # Initialize surrogate from sklearn tree structure
    sk = tree_oracle.tree_
    surrogate = ParametricSoftTree(sk, num_features=X_train.shape[1], num_classes=len(np.unique(tree_oracle.predict(X_train))), init_from_sklearn=True)
    X_acc = []
    y_acc = []
    results = []
    for budget in query_budgets:
        to_add = budget - len(X_acc)
        if to_add > 0:
            new_q = generate_focus_queries(surrogate, X_train, n_queries=to_add, batch_size=batch_size)
            # label with oracle probabilities
            yq = tree_oracle.predict_proba(new_q)
            X_acc.extend(new_q.tolist())
            y_acc.extend(yq.tolist())
        # train surrogate on accumulated queries (probabilities)
        if len(X_acc) > 0:
            train_surrogate_on_queries(surrogate, np.array(X_acc), np.array(y_acc), epochs=5, batch_size=batch_size)
        # reconstruct discrete tree from accumulated queries
        recon = reconstruct_tree_from_queries(np.array(X_acc), np.array(y_acc), max_depth=tree_oracle.get_depth())
        # evaluate fidelity on test set (agreement on predicted class)
        y_or = tree_oracle.predict(X_test)
        y_re = recon.predict(X_test)
        fidelity = (y_or == y_re).mean() if len(y_re) > 0 else 0.0
        print(f'budget={budget} queries={len(X_acc)} fidelity={fidelity:.4f}')
        results.append({'budget': budget, 'n_queries': len(X_acc), 'fidelity': fidelity})
    return surrogate, recon, results, np.array(X_acc), np.array(y_acc)


In [ ]:
surrogate_trained, recon, results, X_q, y_q = run_reconstruction_experiment(tree, X_train, X_test, query_budgets=[20,50], batch_size=16)
print(results)

In [ ]:
# --- Experiments: Iris reconstruction (plots) ---
import matplotlib.pyplot as plt
# run a fuller iris experiment (small-ish budgets) and plot fidelity vs budget
budgets = [20, 50, 100, 200]
surrogate_iris, recon_iris, results_iris, Xq_iris, yq_iris = run_reconstruction_experiment(tree, X_train, X_test, query_budgets=budgets, batch_size=16, seed_size=20)
# extract fidelities
fids = [r['fidelity'] for r in results_iris]
qs = [r['n_queries'] for r in results_iris]
plt.figure(figsize=(6,4))
plt.plot(qs, fids, marker='o', label='Iris')
plt.xlabel('Number of queries')
plt.ylabel('Fidelity')
plt.title('Reconstruction fidelity vs queries (Iris)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.savefig('reconstruction_fidelity_iris.svg')
plt.show()
print('Iris results:', results_iris)


In [ ]:
# --- Experiments: MNIST reconstruction (small / fast variant) ---
# NOTE: MNIST experiments can be slow. This cell uses a small subset and shallow tree for a fast smoke test.
from tensorflow.keras.datasets import mnist
# load and preprocess a small subset
(x_train_full, y_train_full), (x_test_full, y_test_full) = mnist.load_data()
# use a small subset to keep runtime low
n_train = 2000
n_test = 500
X_m_train = x_train_full[:n_train].reshape(n_train, -1).astype('float32') / 255.0
y_m_train = y_train_full[:n_train].astype('int32')
X_m_test = x_test_full[:n_test].reshape(n_test, -1).astype('float32') / 255.0
y_m_test = y_test_full[:n_test].astype('int32')
# train a shallow oracle tree on the subset
mnist_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
mnist_tree.fit(X_m_train, y_m_train)
# run the reconstruction experiment with modest budgets
mnist_budgets = [50, 100, 200]
surrogate_mnist, recon_mnist, results_mnist, Xq_mnist, yq_mnist = run_reconstruction_experiment(mnist_tree, X_m_train, X_m_test, query_budgets=mnist_budgets, batch_size=32, seed_size=50)
# plot results
fids_m = [r['fidelity'] for r in results_mnist]
qs_m = [r['n_queries'] for r in results_mnist]
plt.figure(figsize=(6,4))
plt.plot(qs_m, fids_m, marker='o', color='orange', label='MNIST (subset)')
plt.xlabel('Number of queries')
plt.ylabel('Fidelity')
plt.title('Reconstruction fidelity vs queries (MNIST subset)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.savefig('reconstruction_fidelity_mnist.svg')
plt.show()
print('MNIST (subset) results:', results_mnist)
